In [ ]:
%gui qt
import json
import numpy as np
import fastplotlib as fpl

import pygfx # for typing?

c:\Users\Jan\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastplotlib\layouts\_plot_area.py:342: UserWarning: Could not resolve argspec of Subplot animation function: <function update_preview at 0x000001A0CC2EE3E0>, calling it without arguments.
  warn(
Draw error
Traceback (most recent call last):
  File "c:\Users\Jan\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastplotlib\layouts\_plot_area.py", line 338, in _call_animate_functions
    fn(self)
  File "C:\Users\Jan\AppData\Local\Temp\ipykernel_13672\823770973.py", line 27, in update_preview
    subplot["preview"].data = arr
    ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Jan\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastplotlib\graphics\image.py", line 214, in data
    self._data[:] = data
    ~~~~~~~~~~^^^
  File "c:\Users\Jan\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastplotlib\graphics\features\_base.py", line 336, in set_value_wrapper
    raise exc  # set_value has raised. 

In [2]:
file_path = "diatribes_shaders.json"
# file_path = "jakel101_shaders.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(len(data["shaders"]))

483


In [3]:
# as placeholders, untill we have a single "custom" shape?

# https://docs.pygfx.org/stable/_autosummary/materials/pygfx.materials.PointsMarkerMaterial.html
def map_shapes(tags) -> str:
    if "tunnel" in tags:
        return "💎"
    elif "clouds" in tags:
        return "❤️"
    else:
        return "●"

# is there a proper reference?
def map_colors(tags) -> str:
    if "raymarching" in tags:
        return "blue"
    elif "raymarch" in tags:
        return "purple"
    elif "fractal" in tags:
        return "green"
    else:
        return "red"

In [4]:
# extract data into python objects
username = data["userName"]
dates = [int(shader_data["info"]["date"]) for shader_data in data["shaders"]]
likes = [shader_data["info"]["likes"] for shader_data in data["shaders"]]
views = [shader_data["info"]["viewed"] for shader_data in data["shaders"]]
ids = [shader_data["info"]["id"] for shader_data in data["shaders"]]
shapes = [map_shapes(shader_data["info"]["tags"]) for shader_data in data["shaders"]]
colors = [map_colors(shader_data["info"]["tags"]) for shader_data in data["shaders"]]
info = np.array([dates, likes, views]).T
info

array([[1736386427,          2,        113],
       [1736388271,          8,        202],
       [1736709688,          2,        133],
       ...,
       [1762890910,         16,        142],
       [1762914246,          8,         90],
       [1762826397,         19,        117]], shape=(483, 3))

In [5]:
# for the preview
from wgpu_shadertoy import Shadertoy
def load_shadertoy(shader_data: dict, resolution=(256, 256)) -> Shadertoy:
    shader_data["info"]["username"] = username
    shader_data = scrape_to_api(shader_data)
    kwargs = {"offscreen": True, "resolution": resolution}
    st = Shadertoy.from_json(shader_data, **kwargs)
    return st

# from shadertoys-dataset/download.py https://github.com/Vipitis/shadertoys-dataset/blob/main/download.py
def scrape_to_api(json_data: dict) -> dict:
    """
    transform the dict to be exactly like the API return would provide it
    """
    privacy_keys = {0: "Private", 1: "Public", 2: "Unlisted", 3: "Public API", 4: "Anonymous"}
    
    shader_data = {
        "Shader": {
            "info": json_data["info"],
            "ver": json_data["ver"],
            "renderpass": json_data["renderpass"],
        }
    }
    # del shader_data["Shader"]["info"]["usePreview"] # indicates if a shader is "heavy" and should not be rendered in preview. Maybe useful for filtering?
    for rp in shader_data["Shader"]["renderpass"]:
        for inp in rp["inputs"]:
            inp["src"] = inp.pop("filepath")
            inp["ctype"] = inp.pop("type")

    # TODO: that seems be be incorrect, download gives these. scrape and API gives numbers -.-
    shader_data["Shader"]["info"]["published"] = privacy_keys.get(shader_data["Shader"]["info"]["published"], "Unknown")

    return shader_data


In [ ]:
%gui qt
# https://www.fastplotlib.org/ver/dev/_gallery/scatter/scatter.html#sphx-glr-gallery-scatter-scatter-py
figure = fpl.Figure(size=(1400, 1000), shape=(2, 1))
# TODO: title?
scatter_plot = figure[0, 0].add_scatter(
    data=info[:, :2],
    sizes=(info[:, 2] / 100) + 4, #sorta a min scale
    colors=colors,
    markers=shapes,
    alpha=0.4,
    uniform_edge_color=False,
)
figure[0, 0].auto_scale(maintain_aspect=False)

# have the shadertoy running in the lower half
PREVIEW_RES = (512, 512) # TODO: get this from where? # TODO: flipped axis
random_frame = np.random.randint(0, 256, (*PREVIEW_RES, 3), dtype=np.uint8)
preview = figure[1, 0].add_image(random_frame, name="preview")

def update_preview(subplot):
    shadertoy._update() # manually do this
    i_time = shadertoy._uniform_data["time"]
    next_mem = shadertoy.snapshot(i_time)
    arr = np.asarray(next_mem)
    arr = arr[:, :, :3]
    # TODO: avoid the round trip to ram
    subplot["preview"].data = arr

figure[1, 0].add_animations(update_preview)

shadertoy = load_shadertoy(data["shaders"][0], PREVIEW_RES)


# pick the closest point
@scatter_plot.add_event_handler("click")
def on_pick(ev: pygfx.PointerEvent):
    global shadertoy
    index = ev.pick_info["vertex_index"]
    shader_id = ids[index]
    scatter_plot.edge_colors = "black"  # reset all
    scatter_plot.edge_colors[index] = "white"
    shadertoy = load_shadertoy(data["shaders"][index], PREVIEW_RES)
    print("loading shader:", shadertoy)
    # preview.canvas = shadertoy._canvas

    # print("picked on shader:", shader_id, url)


figure.show()

In [7]:
# qt also closes gracefully
# figure.close()